# AnimationStudio - ComfyUI generation on Google Colab (T4)

This notebook installs the studio plus ComfyUI on a Colab GPU runtime
(Tesla T4, 16 GB VRAM), downloads the model, runs Phase-1 generation,
exports real PNGs, launches the Review UI, and syncs approved output
to GitHub (or downloads it manually).

## Drive budget (free tier = 5 GB)

- **Drive holds exactly one file**: `catalog.db` (the shortlisted/approved
  asset state). Nothing else is written to Drive.
- **Models live on the Colab disk** (`/content/models/`), NOT Drive — a Flux
  model is 12-14 GB and cannot fit a free account. They are re-downloaded
  after a VM reset (`wget -c` resumes). If you upgrade Drive, set
  `CACHE_MODELS_IN_DRIVE = True` in Cell 1 to keep them cached.
- **Generated images are exported into the Colab checkout** and then either
  pushed to GitHub or downloaded manually (Cell 11). No image bytes touch
  Drive.

## Branch = model flavor

| Branch | Model | Notes |
| --- | --- | --- |
| `colab-gpu` | fp8 Flux dev bundle (`flux1-dev.safetensors`, ~12 GB) | Best quality on the T4. One-file `CheckpointLoaderSimple`. |
| `master` | Q4 GGUF (`flux1-dev-Q4_K_S.gguf` + encoders/VAE, ~14 GB) | CPU-grade; also runs on the T4. Needs `ComfyUI-GGUF`. |

## Steps

1. Runtime -> Change runtime type -> T4 GPU (or better).
2. In Cell 1 set `REPO_URL` to your GitHub clone URL.
3. Runtime -> Run all.


In [ ]:
#@title 1. Settings

import os
import subprocess
import sys

# GitHub clone URL for this studio (push master + colab-gpu there first).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}

# colab-gpu -> fp8 Flux (16GB VRAM, best on T4).  master -> Q4 GGUF (CPU-grade).
BRANCH = "colab-gpu"  #@param ["colab-gpu", "master"]

COMFYUI_PORT = 8188  #@param {type:"integer"}
UI_PORT = 8000  #@param {type:"integer"}

# The ONLY thing stored on Google Drive (free tier = 5 GB): the asset DB.
# Shortlisted/approved state survives session resets here.
DRIVE_ROOT = "/content/drive/MyDrive/AnimationStudio"  #@param {type:"string"}
DB = f"{DRIVE_ROOT}/catalog.db"

# The 12-14 GB model cache. Free Drive cannot hold it -> keep on the Colab
# disk (re-downloaded after a VM reset). Set True only if you have space.
CACHE_MODELS_IN_DRIVE = False  #@param {type:"boolean"}

# Generation scope.  Keep small on the free tier (~2 min/image on a T4).
# NOTE: --asset-types takes a COMMA list (no spaces).
GENERATION_ARGS = '--characters "Lily Bunny" --asset-types expressions,poses --count 2 --shortlist 1 --fast-scoring'  #@param {type:"string"}
EXPORT_SCOPE = "characters"  #@param ["characters", "environments", "vehicles", "backgrounds", "props", "all"]
EXPORT_TYPES = ""  #@param {type:"string"}
EXPORT_SIZE = 1024  #@param {type:"integer"}

START_TUNNEL = True  #@param {type:"boolean"}

# Cell 11: push exported/approved output back to GitHub. Off -> download a zip.
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}

# GitHub fine-grained/classic PAT (Settings -> Developer settings -> Tokens).
# Needs Contents: Read+Write on the repo.  Leave empty if the repo is public
# AND your push works without auth; Colab usually has no credential helper,
# so a PAT is required to push from here.
GITHUB_TOKEN = ""  #@param {type:"string"}

# Auto-upload: with --sync-every-image (Cell 9), every image is pushed to
# GitHub the moment it is generated — a lost session never loses more than
# the single in-flight image.  Requires GITHUB_TOKEN for a private repo.
AUTO_SYNC_AFTER_GENERATION = True  #@param {type:"boolean"}

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"
COMFY = f"{WORK}/comfyui"

if REPO_URL.startswith("https://github.com/YOUR_ORG/"):
    raise SystemExit("Set REPO_URL in Cell 1 to your GitHub repository before running.")


In [ ]:
#@title 2. Mount Google Drive (catalog.db only)

from google.colab import drive

drive.mount("/content/drive", force_remount=False)
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive ready (holds catalog.db only):", DRIVE_ROOT)


In [ ]:
#@title 3. Clone repo and install the studio

def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


os.chdir(WORK)
if not os.path.isdir(REPO):
    # Full clone (not --depth 1) so Cell 11 can push approved output back.
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

# Studio first (torch is already preinstalled on Colab), then light deps.
run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "jinja2", "aiosqlite", "python-multipart",
     "pydantic", "scikit-learn", "timm", "diffusers", "transformers"])
print("Studio installed (branch:", BRANCH, ")")


In [ ]:
#@title 4. Install ComfyUI (and ComfyUI-GGUF on the GGUF branch)

if not os.path.isdir(COMFY):
    run(["git", "clone", "--depth", "1",
         "https://github.com/comfyanonymous/ComfyUI.git", COMFY])
run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{COMFY}/requirements.txt"])

if BRANCH == "master":
    gguf = f"{COMFY}/custom_nodes/ComfyUI-GGUF"
    if not os.path.isdir(gguf):
        run(["git", "clone", "--depth", "1",
             "https://github.com/city96/ComfyUI-GGUF.git", gguf])
    run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{gguf}/requirements.txt"])
print("ComfyUI ready at", COMFY)


In [ ]:
#@title 5. Download models (Colab disk, NOT Drive)

MODELS = {
    "colab-gpu": {
        "checkpoints/flux1-dev.safetensors":
            "https://huggingface.co/Comfy-Org/flux1-dev/resolve/main/flux1-dev-fp8.safetensors",
    },
    "master": {
        "checkpoints/flux1-dev-Q4_K_S.gguf":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/flux1-dev-Q4_K_S.gguf",
        "clip/clip_l.safetensors":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/clip_l.safetensors",
        "clip/t5xxl_fp16.safetensors":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/t5xxl_fp16.safetensors",
        "vae/ae.safetensors":
            "https://huggingface.co/city96/FLUX.1-dev-gguf/resolve/main/ae.safetensors",
    },
}[BRANCH]

import shutil

# Default: Colab ephemeral disk (fits free tier, re-downloaded after reset).
# Opt-in: Drive cache, but only if your Drive actually has ~14 GB free.
cache_root = f"{DRIVE_ROOT}/models" if CACHE_MODELS_IN_DRIVE else f"{WORK}/models"

for rel, url in MODELS.items():
    cached = f"{cache_root}/{rel}"
    link = f"{COMFY}/models/{rel}"
    if not (os.path.exists(cached) and os.path.getsize(cached) > 0):
        os.makedirs(os.path.dirname(cached), exist_ok=True)
        print(f"Downloading {rel} ...")
        run(["wget", "-q", "-c", "-O", cached, url])
    os.makedirs(os.path.dirname(link), exist_ok=True)
    if os.path.lexists(link) and not os.path.islink(link):
        os.remove(link)
    if not os.path.islink(link):
        try:
            os.symlink(cached, link)
        except OSError:
            shutil.copyfile(cached, link)
    print(f"OK {rel} ({os.path.getsize(cached) / 1e9:.2f} GB)")

print("Model cache:", cache_root)


In [ ]:
#@title 6. Start the ComfyUI server (auto-restart helper)

# Uses colab/comfy_helpers.py so the generate cell can restart the server
# after a Colab VM recycle instead of failing with Connection refused.
import sys
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)
print("ComfyUI ready on :" + str(COMFYUI_PORT))


In [ ]:
#@title 7. Launch the Review UI and tunnel (monitor generation)

# Started right after ComfyUI so you can watch jobs and view images live
# while Cell 9 generates them. Re-run this cell if you restart the tunnel.

from src.review_ui.app import create_app
from src.universe.batch_generator import resolve_backend

app = create_app(
    db_path=DB,
    generation_backend=resolve_backend("comfyui", comfyui_url=f"http://localhost:{COMFYUI_PORT}"),
    universe_dir=f"{REPO}/Universe",
    world_dir=f"{REPO}/World",
    assets_dir=f"{REPO}/Assets",
    persist_generated_images=True,
)

import socket
import threading
import time
import uvicorn

# Skip rebinding when the UI from an earlier run is still listening.
ui_alive = False
probe = socket.socket()
probe.settimeout(2)
try:
    probe.connect(("127.0.0.1", UI_PORT))
    ui_alive = True
except Exception:
    ui_alive = False
finally:
    probe.close()

if ui_alive:
    print(f"Review UI already running on :{UI_PORT}")
else:
    config = uvicorn.Config(app, host="0.0.0.0", port=UI_PORT, log_level="warning")
    threading.Thread(target=uvicorn.Server(config).run, daemon=True).start()
    print(f"Review UI starting on :{UI_PORT} ...")

if START_TUNNEL:
    import re
    import shutil

    # localtunnel needs Node.js/npm, which Colab does not always ship.  Install
    # once, then use the cached `lt` binary directly (no npx re-fetch each run).
    if not shutil.which("lt"):
        run(["apt-get", "install", "-y", "-qq", "nodejs", "npm"])
        run(["npm", "install", "-g", "--silent", "localtunnel"])
    lt = shutil.which("lt") or "lt"

    def open_tunnel(port, name):
        out = open(f"{WORK}/{name}.log", "w")
        return subprocess.Popen(
            [lt, "--port", str(port)],
            stdout=out, stderr=subprocess.STDOUT,
        )

    p1 = open_tunnel(COMFYUI_PORT, "tunnel_comfyui")
    p2 = open_tunnel(UI_PORT, "tunnel_ui")

    # Poll up to ~60s: the URL only appears once localtunnel connects to its
    # relay, and the first run also pays the npm download cost.
    urls = {}
    for _ in range(30):
        time.sleep(2)
        for name in ("tunnel_comfyui", "tunnel_ui"):
            txt = open(f"{WORK}/{name}.log").read()
            found = re.findall(r"https://[a-z0-9-]+\.loca\.lt", txt)
            if name not in urls or not urls[name]:
                urls[name] = found
        if urls.get("tunnel_comfyui") and urls.get("tunnel_ui"):
            break

    for name in ("tunnel_comfyui", "tunnel_ui"):
        found = urls.get(name) or []
        print(name, "->", found if found else "no URL yet")
        if not found:
            tail = open(f"{WORK}/{name}.log").read().splitlines()[-3:]
            print("   log tail:", tail)
    print()
    print("Review UI:", urls.get("tunnel_ui"))
    print("ComfyUI:  ", urls.get("tunnel_comfyui"))


In [ ]:
#@title 8. Verify the GPU

import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))


In [ ]:
#@title 9. Run Phase-1 generation (ComfyUI backend) — one image, upload, next

# Reuse the running server; restart it if the VM recycled between cells.
import sys
sys.path.insert(0, f"{REPO}/colab")
from comfy_helpers import ensure_comfyui_up

ensure_comfyui_up(port=COMFYUI_PORT, work=WORK, comfy_dir=COMFY)

os.chdir(REPO)
# --sync-every-image pushes each individual image (plus catalog.db) to GitHub
# the moment it is generated, then starts the next one.  A Colab termination
# therefore loses at most the single in-flight image.  Re-running this cell
# skips everything already generated (skip_scored=True) and continues.
!python scripts/generate_phase1_library.py --backend comfyui --comfyui-url http://localhost:{COMFYUI_PORT} --db {DB} --universe {REPO}/Universe --world {REPO}/World --assets {REPO}/Assets --persist-images --sync-every-image --sync-branch {BRANCH} --sync-token {GITHUB_TOKEN} --sync-remote-url {REPO_URL} --sync-git-name {GIT_NAME} --sync-git-email {GIT_EMAIL} {GENERATION_ARGS}

# Per-image sync already happened inside the script (--sync-every-image), so
# every completed image is on GitHub before the next one starts.  This final
# push is just a safety net for any stragglers.
if AUTO_SYNC_AFTER_GENERATION:
    from git_sync import auto_sync
    auto_sync(repo=REPO, branch=BRANCH, db_path=DB, token=GITHUB_TOKEN,
              remote_url=REPO_URL,
              git_name=GIT_NAME, git_email=GIT_EMAIL)


In [ ]:
#@title 10. Export real PNGs into the Colab file tree (not Drive)

os.chdir(REPO)
etype = f"--asset-types {EXPORT_TYPES}" if EXPORT_TYPES else ""
!python scripts/export_assets.py --db {DB} --backend comfyui --comfyui-url http://localhost:{COMFYUI_PORT} --scope {EXPORT_SCOPE} {etype} --size {EXPORT_SIZE} --universe {REPO}/Universe --world {REPO}/World --assets {REPO}/Assets

if AUTO_SYNC_AFTER_GENERATION:
    from git_sync import auto_sync
    auto_sync(repo=REPO, branch=BRANCH, db_path=DB, token=GITHUB_TOKEN,
              remote_url=REPO_URL,
              git_name=GIT_NAME, git_email=GIT_EMAIL)


In [ ]:
#@title 11. Sync approved output (GitHub push or manual download)

# Approve/lock assets in the Review UI (Cell 7 tunnel), then run this cell.
# With AUTO_SYNC_AFTER_GENERATION on, Cells 9-10 already pushed everything;
# this cell is the final safety sync.  Models never leave the Colab disk.
from datetime import datetime

if SYNC_TO_GITHUB:
    sys.path.insert(0, f"{REPO}/colab")
    from git_sync import auto_sync

    auto_sync(repo=REPO, branch=BRANCH, db_path=DB, token=GITHUB_TOKEN,
              remote_url=REPO_URL,
              git_name=GIT_NAME, git_email=GIT_EMAIL,
              message=f"Phase 1 output {datetime.now():%Y-%m-%d %H:%M}")
else:
    import zipfile
    from google.colab import files

    zip_path = f"{WORK}/phase1_export.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{REPO}/catalog.db", "catalog.db")
        for root, _dirs, names in os.walk(f"{REPO}/Assets"):
            for name in names:
                full = os.path.join(root, name)
                z.write(full, os.path.relpath(full, REPO))
    files.download(zip_path)
    print("Downloaded phase1_export.zip (catalog.db + Assets).")


## Next steps

- **Drive holds only `catalog.db`** (shortlisted/approved state). Models live
  on the Colab disk; exported images live in the repo checkout.
- **Approve assets in the Review UI** (Cell 7 tunnel), then re-run Cell 11 to
  push the updated `catalog.db` + exported PNGs to GitHub, or download a zip.
- Extend the library: change `GENERATION_ARGS` in Cell 1, re-run Cells 9-11.
  Already-shortlisted variants are skipped (idempotent).
- After a VM reset: models re-download (Cell 5), `catalog.db` comes back from
  Drive, and previously approved assets are pulled from GitHub.

## Troubleshooting

- CUDA out of memory: shrink the scope in Cell 1, or switch to the L4/A100
  runtime; alternatively set `BRANCH = "master"` (GGUF).
- Model download stalls: re-run Cell 5 (`wget -c` resumes into the cache).
- ComfyUI failed to start: read the log tail printed by Cell 6.
- Drive full: only `catalog.db` should be on Drive. If an older run cached
  `models/` under `DRIVE_ROOT`, delete it to reclaim ~14 GB.
- `git push` fails: Cell 11 pushes with the `REPO_URL` credentials — for
  HTTPS use a personal access token (not your password) when prompted.
